In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.9 MB/s eta 0:00:00


In [8]:
import os
import torch
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

# Define fixed parameters (previously from argparse)
data = "/content/data.yaml"  # Path to your dataset YAML file
epochs = 30  # Number of training epochs
imgsize = 640  # Image size for training
batch = 8  # Batch size
project = "runs/train"  # Project directory
name = ""  # Experiment name
resume = False  # Set to True to resume from checkpoint
weights = "yolo11n.pt"  # Initial weights file

# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Load model and move to device
model = YOLO(weights).to(device)

# Check for resuming from checkpoint
run_dir = os.path.join(project, name)
if resume and os.path.exists(os.path.join(run_dir, "weights", "last.pt")):
    weights = os.path.join(run_dir, "weights", "last.pt")
    model = YOLO(weights).to(device)

# Train the model
results = model.train(
    data=data,
    epochs=epochs,
    imgsz=imgsize,
    batch=batch,
    project=project,
    name=name,
    save=True,
    save_period=1
)

# Plot and save training results
csv_path = os.path.join(results.save_dir, 'results.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    print("\n📊 Training Summary:")
    print(df.tail(1))

    # Plot training losses
    plt.figure(figsize=(10, 6))
    plt.plot(df['epoch'], df['train/box_loss'], label="Box Loss")
    plt.plot(df['epoch'], df['train/cls_loss'], label="Class Loss")
    plt.plot(df['epoch'], df['train/dfl_loss'], label="DFL Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Losses")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(results.save_dir, "loss_plot.png"))
    plt.close()

    # Plot validation metrics
    plt.figure(figsize=(10, 6))
    plt.plot(df['epoch'], df['metrics/precision(B)'], label="Precision")
    plt.plot(df['epoch'], df['metrics/recall(B)'], label="Recall")
    plt.plot(df['epoch'], df['metrics/mAP50(B)'], label="mAP50")
    plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label="mAP50-95")
    plt.xlabel("Epoch")
    plt.ylabel("Metrics")
    plt.title("Validation Metrics")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(results.save_dir, "metrics_plot.png"))
    plt.close()
else:
    print("Could not find results.csv to plot metrics.")

engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=runs/train, rect=False, resume=False, retina_masks=False, save=

Overriding model.yaml nc=80 with nc=12

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytic

YOLO11n summary: 181 layers, 2,592,180 parameters, 2,592,164 gradients, 6.5 GFLOPs

Transferred 448/499 items from pretrained weights
Freezing layer 'model.23.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 765.0±298.2 MB/s, size: 30.8 KB)


train: Scanning /content/train/train/labels... 6137 images, 4 backgrounds, 0 corrupt: 100%|██████████| 6137/6137 [00:02<00:00, 2568.11it/s]


train: New cache created: /content/train/train/labels.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 183.9±38.7 MB/s, size: 14.6 KB)


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      30/30      1.29G     0.4462     0.2176     0.9124          1        640: 100%|██████████| 768/768 [01:58<00:00,  6.48it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 15.55it/s]


                   all          3          3          1          1      0.995      0.863

30 epochs completed in 1.010 hours.
Optimizer stripped from runs/train/train3/weights/last.pt, 5.5MB
Optimizer stripped from runs/train/train3/weights/best.pt, 5.5MB

Validating runs/train/train3/weights/best.pt...
Ultralytics 8.3.182 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n summary (fused): 100 layers, 2,584,492 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 24.28it/s]


                   all          3          3          1          1      0.995      0.951
                  bird          3          3          1          1      0.995      0.951
Speed: 0.3ms preprocess, 7.8ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/train/train3

📊 Training Summary:
    epoch    time  train/box_loss  train/cls_loss  train/dfl_loss  \
29     30  3634.4         0.44621         0.21756          0.9124   

    metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  \
29                   1.0                1.0             0.995   

    metrics/mAP50-95(B)  val/box_loss  val/cls_loss  val/dfl_loss    lr/pg0  \
29              0.86272       0.48803       0.21693       0.81954  0.000027   

      lr/pg1    lr/pg2  
29  0.000027  0.000027  


In [3]:
!apt-get install unrar

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [4]:
!unrar x /content/train.rar /content/train/
!unrar x /content/valid.rar /content/valid/
!unrar x /content/test.rar /content/test/

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
Extracting  /content/train/train/labels/dog_ccfe7d56-5f0d-11ed-a3e2-c894027f4afb_jpg.rf.278076ccc093cea29ab49f6722a07332.txt      99%  OK 
Extracting  /content/train/train/labels/dog_cd0d6c4e-5f0d-11ed-b8c6-c894027f4afb_jpg.rf.6e96ca0f05c5c75cd2dd3c527abdabd9.txt      99%  OK 
Extracting  /content/train/train/labels/dog_cd1c5a6a-5f0d-11ed-be4a-c894027f4afb_jpg.rf.fb05e8640ed0b93dad3c5d8c9e74e13b.txt      99%  OK 
Extracting  /content/train/train/labels/dog_cd2b0024-5f0d-11ed-a31c-c894027f4afb_jpg.rf.135b924602799fbe5314912a2a902682.txt      99%  OK 
Extracting  /content/train/train/labels/dog_cd3a1122-5f0d-11ed-abb4-c894027f4afb_jpg.rf.7da622965380edd97411f4244f9161c4.txt      99%  OK 
Extracting  /content/train/train/labels/dog_cd48fda4-5f0d-11ed-919e-c894027f4afb_jpg.rf.be1c6d1c4649ea4258cec124f23cb8aa.txt      99%  OK 
Extracting  /content/train/train/labels/dog_cd588fee-5f